In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import login, HfApi

# 1. Load variables from your local .env file
load_dotenv()

# 2. Retrieve the token from the environment
token = os.getenv("HF_TOKEN")

# 3. Authenticate and verify access
login(token=token)

api = HfApi(token=token)
info = api.model_info("meta-llama/Meta-Llama-3.1-8B")

print("Model access verified:", info.id)


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Model access verified: meta-llama/Llama-3.1-8B


In [ ]:
import os
from dotenv import load_dotenv

# 1. Securely handle the Jupyter Notebook folder offset on local Windows
NOTEBOOK_DIR = os.getcwd()  # This will point to '...\repro...\notebooks'

if os.path.basename(NOTEBOOK_DIR) == "notebooks":
    # Step up one level to the true project root folder
    PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
else:
    PROJECT_ROOT = NOTEBOOK_DIR

# 2. Automatically load your .env token at runtime from the root workspace
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

# 3. Correctly route paths relative to your true root folder
SAVE_DIR = os.path.join(PROJECT_ROOT, "results", "accuracy")
os.makedirs(SAVE_DIR, exist_ok=True)

EVAL_SCRIPT = os.path.join(PROJECT_ROOT, "src_patching", "mcqa_evaluator.py")
SUMMARY_FILE = os.path.join(SAVE_DIR, "run_summary.csv")



In [5]:
import gc
import subprocess
import sys
import csv
import os
import torch

# 1. Windows Local Output directory (creates 'results/accuracy' in your current working folder)
BASE_DIR = os.getcwd()
SAVE_DIR = os.path.join(BASE_DIR, "results", "accuracy")
os.makedirs(SAVE_DIR, exist_ok=True)

# 2. Path to your local evaluation script (Windows-safe relative path)
EVAL_SCRIPT = os.path.join("src_patching", "mcqa_evaluator.py")

# Belebele languages
languages = [
    "eng_Latn",
    "fra_Latn",
    "deu_Latn",
    "hin_Deva",
    "ita_Latn",
    "por_Latn",
    "spa_Latn",
    "tha_Thai",
]

MODEL = "Llama3.1"
DATASET = "belebele"
SUMMARY_FILE = os.path.join(SAVE_DIR, "run_summary.csv")

def run_experiment(model, dataset, language):
    command = [
        sys.executable,
        EVAL_SCRIPT,
        "--dataset", dataset,
        "--lang", language,
        "--llm_name", model,
        "--mc_key", "ABCD",
        "--batch_size", "1",
        "--save_dir", SAVE_DIR,
    ]

    try:
        print(f"\nRunning {dataset} | {language} | {model}")
        # Note: Added shell=True for better execution stability across certain Windows environments
        result = subprocess.run(command, check=False)
        print(f"Exit code: {result.returncode}")
        return result.returncode

    except Exception as e:
        print(f"Error running {language}: {e}")
        return -1

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("Cleanup completed.")


# Load previous progress so successful languages are skipped
if os.path.exists(SUMMARY_FILE):
    with open(SUMMARY_FILE, newline="", encoding="utf-8") as f:
        results = list(csv.DictReader(f))
else:
    results = []

completed = {
    row["language"]
    for row in results
    if row["dataset"] == DATASET
    and row["model"] == MODEL
    and row["status"] == "SUCCESS"
}

for i, language in enumerate(languages, start=1):
    if language in completed:
        print(f"Skipping {language}: already successful.")
        continue

    print(f"\n{'=' * 55}")
    print(f"Experiment {i}/{len(languages)}: {language}")
    print(f"{'=' * 55}")

    exit_code = run_experiment(MODEL, DATASET, language)
    status = "SUCCESS" if exit_code == 0 else "FAILED"

    # Replace any previous record for this experiment
    results = [
        row for row in results
        if not (
            row["dataset"] == DATASET
            and row["model"] == MODEL
            and row["language"] == language
        )
    ]

    results.append({
        "dataset": DATASET,
        "model": MODEL,
        "language": language,
        "status": status,
        "exit_code": exit_code,
    })

    # Save progress after each language (Added explicit UTF-8 encoding for Windows compatibility)
    with open(SUMMARY_FILE, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "dataset", "model", "language",
                "status", "exit_code"
            ],
        )
        writer.writeheader()
        writer.writerows(results)

    print(f"{language}: {status}")

print("\nAll experiments finished.")
print(f"Summary CSV: {SUMMARY_FILE}")
print(f"Detailed results: {os.path.join(SAVE_DIR, 'Llama3.1', 'belebele')}")



Experiment 1/8: eng_Latn

Running belebele | eng_Latn | Llama3.1
Exit code: 2
Cleanup completed.
eng_Latn: FAILED

Experiment 2/8: fra_Latn

Running belebele | fra_Latn | Llama3.1
Exit code: 2
Cleanup completed.
fra_Latn: FAILED

Experiment 3/8: deu_Latn

Running belebele | deu_Latn | Llama3.1
Exit code: 2
Cleanup completed.
deu_Latn: FAILED

Experiment 4/8: hin_Deva

Running belebele | hin_Deva | Llama3.1
Exit code: 2
Cleanup completed.
hin_Deva: FAILED

Experiment 5/8: ita_Latn

Running belebele | ita_Latn | Llama3.1
Exit code: 2
Cleanup completed.
ita_Latn: FAILED

Experiment 6/8: por_Latn

Running belebele | por_Latn | Llama3.1
Exit code: 2
Cleanup completed.
por_Latn: FAILED

Experiment 7/8: spa_Latn

Running belebele | spa_Latn | Llama3.1
Exit code: 2
Cleanup completed.
spa_Latn: FAILED

Experiment 8/8: tha_Thai

Running belebele | tha_Thai | Llama3.1
Exit code: 2
Cleanup completed.
tha_Thai: FAILED

All experiments finished.
Summary CSV: c:\Users\cream\reproducing-cross-lingual

In [ ]:
import os
import json

result_dir = (
    "/kaggle/working/results/accuracy/"
    "Llama3.1/belebele"
)

for lang in [
    "eng_Latn", "fra_Latn", "deu_Latn", "hin_Deva",
    "ita_Latn", "por_Latn", "spa_Latn", "tha_Thai"
]:
    filename = (
        f"belebele_eval_Llama3.1_{lang}_ABCD.jsonl"
    )
    path = os.path.join(result_dir, filename)

    if not os.path.exists(path):
        print(f"{lang}: MISSING")
        continue

    with open(path, "r") as f:
        records = [json.loads(line) for line in f if line.strip()]

    if records:
        accuracy = sum(r["acc"] for r in records) / len(records)
        print(
            f"{lang}: {len(records)}/900 examples, "
            f"accuracy={accuracy:.4%}"
        )
    else:
        print(f"{lang}: EMPTY FILE")